# Descriptive statistics -- core analysis

**Specification.** `lst_day_mean ~ ntl_harm`, with `ntl_harm` instrumented by
`mine_count_20km` (count of mines within 20 km), pixel and year fixed effects:

```
lst_day_mean ~ 1 | pixel_id + year | (ntl_harm ~ mine_count_20km)
```

`ntl_harm` is ~68 % exact zeros, so we also look at the **extensive vs intensive
NTL margin** (a threshold at `ntl_harm >= 7`, roughly the top decile of all
pixel-years) and at the **mine price-shock** instrument as an alternative to the
count.

| Section | Reads on |
| --- | --- |
| 1 univariate distributions of `lst_day_mean`, `ntl_harm`, the instrument | scale, skew, zero-mass |
| 2 the NTL margin -- extensive vs intensive | can the zero-inflation be split into a participation + an intensity model |
| 3 binscatter `lst_day_mean` vs `ntl_harm` | the OLS conditional mean |
| 4 first stage / reduced form (`mine_count_20km`) | does the instrument move `ntl_harm`; does the outcome move with it |
| 5 mine price-shock instrument | distribution and relevance of `mine_priceshock_20km` |
| 6 distributions split by mine exposure | is the treated sub-sample comparable |
| 7 correlation matrix | collinearity among regressors / mine-radius variants |
| 8 alternative measures -- night LST, GLASS air temp, VIIRS | does the picture survive alternative outcomes and an alternative NTL measure |
| 9 data-quality filters -- valid count, gas flares | how the sample and the NTL tail move when unreliable pixel-years are dropped |

The **spatial** companion to this notebook -- one map per assembled column for a
single tile / year -- is `descriptive_overview.ipynb`.

All statistics are computed in DuckDB by `src.viz`, so the full ~37 M-row panel
never lands in pandas. Cells that fall outside `src.viz` today (rate-by-year,
threshold-sensitivity, simple bar charts) drop to a DuckDB query plus matplotlib
and are marked as candidates for future `viz` helpers.

In [ ]:
import numpy as np
import duckdb
import matplotlib.pyplot as plt

import src.viz as viz

# --- where the assembled panel lives -----------------------------------------
# Local mirror of HPC's data_nobackup/ (see orchestration/configs/data.local.yaml).
PROJECT = "/Users/felixschulz/Library/CloudStorage/OneDrive-Personal/Dokumente/Job/UNI/Basel/Research/growth-and-temperature"
GRID = "10km"
PANEL = f"{PROJECT}/data/assembled/grid={GRID}/shake=base/ix=*/iy=*/*.parquet"

Y = "lst_day_mean"       # outcome
X = "ntl_harm"           # endogenous regressor
Z = "mine_count_20km"    # instrument
PS = "mine_priceshock_20km"  # alternative instrument
NTL_HI = 7               # extensive/intensive NTL threshold (~ p90 of all pixel-years)
VALID_MIN = 300          # min annual valid daytime-LST obs to keep a pixel-year (section 9)

plt.rcParams["figure.dpi"] = 110

In [ ]:
# One shared connection, reused by every plot below (src.viz accepts `con=`/`table=`).
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")  # no per-query widget in the saved notebook

# Materialise the columns the analysis touches into one table. This is the only
# heavy scan (~10 s on the 10 km panel); every plot afterwards hits the in-memory
# table. A `SELECT *` VIEW over the hive glob is avoided on purpose -- `union_by_name`
# re-infers per-part types and a re-planned query can then trip "contents of view
# were altered".
con.execute(f"""
    CREATE OR REPLACE TABLE panel AS
    SELECT
        pixel_id, year,
        lst_day_mean, lst_night_mean, glass_ta_mean,
        ntl_harm, viirs_annual_median AS viirs,
        mine_count_20km, mine_count_10km, mine_count_50km,
        mine_priceshock_10km, mine_priceshock_20km, mine_priceshock_50km,
        valid_month_count_day_annual AS valid_day,
        flare_band,
        ln(ntl_harm + 1.0)                                    AS log_ntl,
        (ntl_harm >= {NTL_HI})                                AS ntl_hi,
        (coalesce(flare_band, 0) > 0)                         AS is_flare,
        (mine_count_20km > 0)                                 AS has_mine,
        max(mine_count_20km > 0) OVER (PARTITION BY pixel_id) AS mine_ever
    FROM read_parquet('{PANEL}', union_by_name = true, hive_partitioning = true)
""")

con.execute("CREATE OR REPLACE VIEW panel_mines     AS SELECT * FROM panel WHERE mine_count_20km > 0")
con.execute(f"CREATE OR REPLACE VIEW panel_intensive AS SELECT * FROM panel WHERE ntl_harm >= {NTL_HI}")
# VIIRS is a raw-sensor NTL series; 2012 is on a different scale, so keep 2013-2021.
con.execute("CREATE OR REPLACE VIEW panel_viirs AS SELECT * FROM panel WHERE year BETWEEN 2013 AND 2021 AND viirs IS NOT NULL")
# GLASS 2m air temperature covers 2000-2020.
con.execute("CREATE OR REPLACE VIEW panel_glass AS SELECT * FROM panel WHERE glass_ta_mean IS NOT NULL")
# The gas-flare mask only exists for VIIRS-era rows (flare_band NOT NULL).
con.execute("CREATE OR REPLACE VIEW panel_flareable AS SELECT * FROM panel WHERE flare_band IS NOT NULL")
con.execute(f"""
    CREATE OR REPLACE VIEW pix_hi AS
    SELECT pixel_id, count(*) AS yrs, sum((ntl_harm >= {NTL_HI})::INT) AS yrs_hi
    FROM panel GROUP BY pixel_id
""")

con.execute("""
    SELECT count(*) AS rows, count(DISTINCT year) AS years,
           avg(has_mine::INT) AS share_mine, avg(ntl_hi::INT) AS share_ntl_hi
    FROM panel
""").fetchdf()

## 1. Univariate distributions

In [ ]:
fig = viz.plot_distribution(con, Y, table="panel", bins=60)
fig.suptitle("Daytime LST (mean), pixel-year", y=1.02)
fig

**Result.** 37.4 M pixel-years, 1.21 M pixels, 1992-2022. `lst_day_mean` mean
303.3 K (~30 degC), sd 13.6 K, median 306.6 K, p1-p99 = 271.9-324.6 K
(-1 to +52 degC), left-skewed (a long cold tail from high-latitude / winter
pixels).

**Interpretation.** 32.5 % of pixel-years have no MODIS LST (cloud gaps, and no
MODIS before 2000); the regression sample is the ~25 M non-null rows. The outcome
is on a clean physical scale with no fill-value spikes -- no trimming needed.

In [ ]:
# NTL is heavily right-skewed and zero-inflated. Raw x, but a log *count* axis
# (logy) makes the decaying tail visible without transforming the variable;
# the faint overlay is log1p(ntl_harm).
fig = viz.plot_distribution(
    con, X, table="panel", transform="identity", overlay_transform="log1p",
    bins=80, logy=True, vlines=NTL_HI,
)
fig.axes[0].set_title("ntl_harm -- raw (log count axis), log1p overlay, dashed line at 7")
fig

**Result.** **67.8 % of pixel-years are exactly 0**; among the positive third the
mass decays smoothly out to 146 (mean 2.9, sd 9.7). Only **10.9 %** of all
pixel-years reach `ntl_harm >= 7` (33.9 % of the positive pixels). On the log
count axis the tail is a near-straight decline -- roughly geometric.

**Interpretation.** A linear `ntl_harm` term is dominated by the zero/non-zero
contrast and its scale is arbitrary; `log1p` (or `asinh`) spreads the positive
mass but leaves the point mass at 0. This is what motivates the
extensive/intensive split in section 2.

In [ ]:
# The instrument: mostly zero, long tail. ECDF makes the mass point legible.
fig = viz.plot_distribution(con, Z, table="panel", kind="ecdf", bins=200)
fig.axes[0].set_title("mine_count_20km -- ECDF (note the mass at 0)")
fig

**Result.** **94.9 % of pixel-years have `mine_count_20km` = 0.** Among the 5.1 %
with a mine (1.90 M rows, 92.4 k pixels): median 0.84, p90 = 2.0, p99 = 6.2,
max 30.

**Interpretation.** The instrument is a rare-event count; all its identifying
variation lives in ~5 % of the panel. The first stage and reduced form below are
therefore run on that subsample (`panel_mines`) -- not a modelling choice, just
where the signal is. Expect wide standard errors and check first-stage strength.

## 2. The NTL margin -- extensive vs intensive

With two-thirds of pixel-years at `ntl_harm = 0`, one option is to split the
regressor into

* an **extensive margin** -- the indicator `1{ntl_harm >= 7}` (is the pixel
  "meaningfully lit"), and
* an **intensive margin** -- the continuous distribution *conditional* on
  `ntl_harm >= 7`.

These plots check whether that split is well-behaved and where its identifying
variation comes from.

In [ ]:
# Threshold sensitivity: how sample share and mean LST move with the cutoff c.
# (DuckDB + matplotlib -- candidate for a future viz.plot_rate_by helper.)
cuts = [1, 2, 3, 5, 7, 10, 15, 20, 30]
ts = con.execute(f"""
    SELECT c,
           avg((ntl_harm >= c)::INT)                            AS share,
           avg(CASE WHEN ntl_harm >= c THEN lst_day_mean END)   AS lst
    FROM panel CROSS JOIN (SELECT unnest({cuts}) AS c) g
    GROUP BY c ORDER BY c
""").fetchdf()

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(ts.c, ts.share, "o-"); ax[0].axvline(NTL_HI, ls="--", c="0.4")
ax[0].set(xlabel="NTL cutoff c", ylabel="share of pixel-years with ntl_harm >= c")
ax[1].plot(ts.c, ts.lst, "o-"); ax[1].axvline(NTL_HI, ls="--", c="0.4")
ax[1].set(xlabel="NTL cutoff c", ylabel="mean lst_day_mean | ntl_harm >= c")
fig.tight_layout(); fig

**Result.** Both curves are smooth in the cutoff -- no kink singles out 7. At
c = 7 the "lit" set is 10.9 % of pixel-years (4.08 M rows); mean LST among lit
pixels falls monotonically from 300.5 K (c >= 1) to ~298.8 K (c >= 20).

**Interpretation.** `NTL_HI = 7` is a reasonable, not a data-mandated, choice --
5 or 10 would give a similar sample and the same qualitative picture. Re-run the
notebook with a different `NTL_HI` to see the split move. The steady "brighter =
cooler" gradient across every cutoff previews the OLS result in section 3.

In [ ]:
fig = viz.plot_distribution(con, X, table="panel_intensive", bins=60, logy=True, vlines=NTL_HI)
fig.axes[0].set_title(f"ntl_harm | ntl_harm >= {NTL_HI}  -- intensive margin (log count axis)")
fig

In [ ]:
# ln(ntl_harm) on the intensive margin -- this is the candidate intensive-margin regressor.
fig = viz.plot_distribution(con, X, table="panel_intensive", transform="log", bins=60)
fig.axes[0].set_title(f"log(ntl_harm) | ntl_harm >= {NTL_HI}  -- roughly symmetric")
fig

**Result.** Conditional on `ntl_harm >= 7`, the level is still heavy-tailed
(mean 23.1, sd 19.7, median 15.2, max 146), **but `log(ntl_harm)` is close to
symmetric** -- mean 2.88, sd 0.68, p10-p90 = 2.09-3.95.

**Interpretation.** The intensive margin gives a clean, near-Gaussian regressor
once you condition on participation and take logs. So the split buys a
well-specified intensity model at the cost of estimating it on ~11 % of the
panel; the extensive margin (next plot) carries the rest.

In [ ]:
# Extensive margin over time: P(ntl_harm >= NTL_HI) by year.
# (DuckDB + matplotlib -- candidate for a future viz.plot_rate_by helper.)
et = con.execute(f"""
    SELECT year, avg((ntl_harm >= {NTL_HI})::INT) AS p FROM panel GROUP BY year ORDER BY year
""").fetchdf()

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.axvspan(2012.5, 2014.5, color="0.88", zorder=0)
ax.plot(et.year, et.p, "o-")
ax.annotate("DMSP -> VIIRS\nharmonization", (2013.5, et.p.max() * 0.92), ha="center", fontsize=8)
ax.set(xlabel="year", ylabel=f"P(ntl_harm >= {NTL_HI})", title="Extensive NTL margin over time")
fig

**Result.** The lit share sits at 7-12 % from 1992 to 2012, **spikes to 21 % in
2013, collapses to ~8.5 % in 2014-2016, then climbs to 12-17 % in 2017-2022**.

**Interpretation.** The 2013 spike / 2014 dip is a harmonization artifact at the
DMSP -> VIIRS transition, not real electrification. Year fixed effects absorb the
common level shift, but a threshold indicator is sensitive to it -- consider
dropping 2013-2014, adding a sensor-era interaction, or checking robustness to
`NTL_HI`. The post-2016 upward trend looks like genuine growth.

In [ ]:
fig = viz.plot_distribution(con, "yrs_hi", table="pix_hi", bins=32)
fig.axes[0].set_title(f"Per-pixel count of years with ntl_harm >= {NTL_HI}  (0-31)")
fig

**Result.** **75.1 % of pixels never cross the threshold, 4.6 % are always above
it, and 20.2 % switch** at least once (244,153 pixels). The histogram is
U-shaped -- piled at 0 and at 31, thin in between.

**Interpretation.** The extensive-margin regression with pixel fixed effects is
identified off those ~244 k switching pixels -- more within-pixel variation than
the mine instrument offers, but still a minority of the grid. The always/never
pixels contribute only to the fixed effects.

## 3. Outcome vs regressor -- the OLS relationship

In [ ]:
fig = viz.plot_binscatter(con, y=Y, x=X, x_transform="log1p", table="panel", bins=30)
fig.axes[0].set_title(f"{Y} vs log1p({X})  --  30 equal-count bins, SE whiskers, OLS line")
fig

**Result.** Downward-sloping: binned mean LST falls from ~305.9 K in the darkest
bin to ~298.8 K in the brightest, OLS slope on the binned points **~-0.53 K per
log point**; raw `corr(lst_day_mean, ntl_harm) = -0.092`.

**Interpretation.** Unconditionally, brighter pixels are a few K *cooler* -- this
is cross-sectional confounding (cities are green / temperate / coastal; hot
bright deserts carry little NTL), not an effect of light. Pixel + year fixed
effects and the instrument are meant to purge exactly this gradient.

## 4. First stage and reduced form (`mine_count_20km`)

Restricted to pixels with at least one mine within 20 km (`panel_mines`), since
the instrument has essentially no variation elsewhere.

In [ ]:
fig = viz.plot_binscatter(con, y=X, x=Z, y_transform="log1p", table="panel_mines", bins=25)
fig.axes[0].set_title(f"First stage: log1p({X}) vs {Z}")
fig

**First stage.** Positive: binned `log1p(ntl_harm)` rises from ~-2.4 (fewest
mines) to ~+1.1 (most), slope on binned points **~+0.67**. But the *within*
pixel + year partial correlation of the instrument with `ntl_harm` is only
**~0.038** (FE first-stage slope ~0.84 in levels).

**Interpretation.** More mines within 20 km do bring more nighttime light, so the
instrument is relevant in sign -- but weakly, and only on the ~5 % mine
subsample. Weak-instrument territory: report the first-stage F and use a
weak-IV-robust CI.

In [ ]:
fig = viz.plot_binscatter(con, y=Y, x=Z, table="panel_mines", bins=25)
fig.axes[0].set_title(f"Reduced form: {Y} vs {Z}")
fig

**Reduced form.** Downward-sloping: binned mean LST falls from ~302.8 K to
~298.1 K across the instrument range, slope **~-0.19 K per mine**.

**Interpretation.** Consistent in sign with the OLS gradient; with a positive
first stage this implies a negative 2SLS coefficient. Small, and -- like the
first stage -- driven by a restricted, non-random set of pixels, so it partly
reflects *where mines are* (cooler uplands, particular regions). This is the main
exclusion-restriction worry.

## 5. Mine price-shock instrument (`mine_priceshock_20km`)

An alternative to the bare count: mine exposure interacted with commodity price
movements, so it varies within a mine pixel over time. Descriptives on the mine
subsample.

In [ ]:
fig = viz.plot_distribution(con, PS, table="panel_mines", bins=60, logy=True)
fig.axes[0].set_title(f"{PS} | mine within 20 km  (log count axis)")
fig

**Result.** Non-negative (0 to 47.5) -- a magnitude, not a signed shock.
**99.0 % of all pixel-years are 0; even within `panel_mines` 79.7 % are 0**
(p90 = 7.2, p99 = 12.1). So the price-shock fires only in mine-pixel-years that
coincide with a commodity price move.

**Interpretation.** Sparser than the count. Its appeal is timing (year-to-year
price variation within a pixel), not coverage; the effective sample is a fraction
of an already-small mine subsample.

In [ ]:
fig = viz.plot_binscatter(con, y=PS, x=Z, table="panel_mines", bins=25)
fig.axes[0].set_title(f"{PS} vs {Z}  (mine pixels)")
fig

**Result.** Increasing but loose -- `corr(mine_priceshock_20km, mine_count_20km)
= 0.37`. Price-shock radii are less collinear than the count radii
(10-20 km r = 0.66, 20-50 km r = 0.58, vs 0.70-0.80 for the counts).

**Interpretation.** The price-shock carries variation the count does not, so it
is a genuine second instrument / over-identifying check rather than a
re-labelling of the same thing.

In [ ]:
# Price-shock over time (mean over mine pixels).
pt = con.execute(f"""
    SELECT year, avg({PS}) AS m, stddev_samp({PS}) AS sd FROM panel_mines GROUP BY year ORDER BY year
""").fetchdf()
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(pt.year, pt.m, "o-")
ax.fill_between(pt.year, pt.m - pt.sd, pt.m + pt.sd, alpha=0.15)
ax.set(xlabel="year", ylabel=f"mean {PS} | mine pixel", title="Price-shock exposure over time")
fig

**Result.** Mean exposure among mine pixels is fairly flat (~1.4-1.6) with a mild
dip through the 2000s and a recovery after 2020; the spread (+/- 1 sd) is large
relative to the mean.

**Interpretation.** No strong secular trend -- the identifying content is the
cross-pixel/year deviations, which year fixed effects leave intact. The wide band
is the zero-inflation showing through.

In [ ]:
fig = viz.plot_binscatter(con, y=X, x=PS, y_transform="log1p", table="panel_mines", bins=25)
fig.axes[0].set_title(f"Price-shock first stage: log1p({X}) vs {PS}")
fig

**Result.** Weakly positive on the binned means. Raw `corr(mine_priceshock_20km,
ntl_harm) = 0.009` -- essentially zero -- vs 0.074 for the count.

**Interpretation.** Any first stage for the price-shock has to come almost
entirely from *within-pixel* variation; the raw association is nil. Its
first-stage F is likely to be weaker than the count's -- treat it as a
robustness instrument and lean on weak-IV-robust inference.

## 6. Distributions split by mine exposure

In [ ]:
# `mine_ever` = the pixel has a mine within 20 km in at least one year.
fig = viz.plot_distribution(con, Y, table="panel", groupby="mine_ever", kind="ecdf", bins=200)
fig.axes[0].set_title(f"{Y} by mine exposure (ever within 20 km)")
fig

**Result.** Mine-ever pixels are **~3.6 K cooler** on average (299.98 vs
303.59 K; medians 300.7 vs 306.9 K); the whole CDF is shifted left.

**Interpretation.** Treated and never-treated pixels are not comparable in levels
-- mines sit in systematically different terrain. Pixel fixed effects are
essential, and the size of this gap (larger than the NTL gap below) is a concrete
reason to stress-test the exclusion restriction.

In [ ]:
fig = viz.plot_distribution(con, "log_ntl", table="panel", groupby="mine_ever", kind="ecdf", bins=200)
fig.axes[0].set_title("log1p(ntl_harm) by mine exposure")
fig

**Result.** Mine-ever pixels are brighter (mean `log_ntl` 0.71 vs 0.49; raw
`ntl_harm` 4.01 vs 2.84), CDF shifted right -- both groups still >60 % zeros.

**Interpretation.** The instrument separates pixels on NTL, consistent with the
first stage, but modestly. The LST gap between the groups exceeds the NTL gap --
a hint that mines correlate with LST partly through non-light channels.

In [ ]:
# Extensive NTL margin by mine exposure (bar).
mb = con.execute(f"""
    SELECT mine_ever, avg((ntl_harm >= {NTL_HI})::INT) AS p, count(*) AS n
    FROM panel GROUP BY mine_ever ORDER BY mine_ever
""").fetchdf()
fig, ax = plt.subplots(figsize=(4.6, 3.5))
ax.bar(["no mine ever", "mine ever"], mb.p, color=["#4c72b0", "#dd8452"])
for i, p in enumerate(mb.p):
    ax.text(i, p, f"{p:.1%}", ha="center", va="bottom", fontsize=9)
ax.set(ylabel=f"P(ntl_harm >= {NTL_HI})", title="Extensive NTL margin by mine exposure")
fig

**Result.** `P(ntl_harm >= 7)` is **15.4 %** for mine-ever pixels vs **10.5 %**
for the rest.

**Interpretation.** Mines shift the extensive margin too, not just the intensity
-- so an extensive-margin IV (`1{ntl_harm >= 7}` instrumented by mine exposure)
is also on the table, and with more switchers than the intensive one it may be
better powered.

## 7. Correlation / collinearity screen

In [ ]:
fig = viz.plot_corr_matrix(
    con,
    [Y, X, Z, "mine_count_10km", "mine_count_50km", PS, "mine_priceshock_50km"],
    table="panel",
    method="pearson",
)
fig

**Result.**

| pair | r |
| --- | --- |
| `lst_day_mean` - `ntl_harm` | -0.092 |
| `ntl_harm` - `mine_count_20km` (raw relevance) | +0.074 |
| `ntl_harm` - `mine_priceshock_20km` | **+0.009** |
| `mine_count_20km` - `mine_count_10km` / `_50km` | **0.80 / 0.70** |
| `mine_count_20km` - `mine_priceshock_20km` | 0.37 |

**Interpretation.** Raw instrument relevance is weak for the count and ~nil for
the price-shock (both need within-FE variation). The three mine-count radii are
highly collinear -- use one (20 km). The price-shock is only moderately
correlated with the count, confirming it as a distinct instrument.

## 8. Alternative temperature and lights measures

Does the OLS picture (section 3) survive alternative **outcomes**
(`lst_night_mean`; `glass_ta_mean`, GLASS 2 m air temperature) and an alternative
**NTL measure** (`viirs_annual_median`, raw VIIRS)?

In [ ]:
fig = viz.plot_distribution(con, "lst_night_mean", table="panel", bins=60)
fig.axes[0].set_title("Night-time LST (mean), pixel-year")
fig

**Result.** Mean 284.4 K (~11 degC), sd 11.2 K (tighter than daytime's 13.6),
p1-p50-p99 = 259.0 / 287.6 / 300.4 K; same 32.5 % missing as day.
`corr(lst_day_mean, lst_night_mean) = 0.85`.

**Interpretation.** Night LST is a coherent second outcome -- correlated with the
day series but not redundant (it removes the solar-loading component and isolates
stored heat).

In [ ]:
fig = viz.plot_binscatter(con, y="lst_night_mean", x=X, x_transform="log1p", table="panel", bins=30)
fig.axes[0].set_title("lst_night_mean vs log1p(ntl_harm)  -- compare to section 3 (daytime)")
fig

**Result.** Still downward-sloping, but flatter: `corr(ntl_harm, lst_night_mean)
= -0.033` vs `-0.092` for daytime. The negative NTL-LST gradient is **weaker at
night**.

**Interpretation.** If a heat-retention / urban-heat-island mechanism dominated,
the association should be *stronger* at night; that it is weaker points to the
daytime gradient being driven mainly by land cover -- bright, vegetated or
irrigated pixels evaporate and cool during the day -- rather than nighttime heat
storage. The sign is robust to the outcome; the magnitude is not.

### GLASS 2 m air temperature (`glass_ta_mean`)

In [ ]:
fig = viz.plot_distribution(con, "glass_ta_mean", table="panel_glass", bins=60)
fig.axes[0].set_title("GLASS 2 m air temperature (annual mean), pixel-year, 2000-2020")
fig

**Result.** Mean 290.7 K (~17.6 degC), sd 10.0 K (tighter than day LST's 13.6),
p1-p50-p99 = 267.8 / 294.3 / 303.6 K. Coverage is 2000-2020 (~98.5 % of pixels
per year; no 1992-1999, no 2021-2022) -- so it shortens the panel by ~8 years at
the start.

**Interpretation.** Air temperature, not skin temperature -- the variable that
actually maps to human exposure and economic outcomes. It is highly correlated
with the LST series (`corr` with day LST 0.92, with night LST **0.98**), so it
carries most of the same signal on a physically more meaningful scale.

In [ ]:
fig = viz.plot_binscatter(con, y="glass_ta_mean", x=X, x_transform="log1p", table="panel_glass", bins=30)
fig.axes[0].set_title("glass_ta_mean vs log1p(ntl_harm)  -- air-temp analogue of section 3")
fig

**Result.** Downward-sloping like the LST binscatter; `corr(ntl_harm,
glass_ta_mean) = -0.060` -- between the daytime (-0.092) and night-time (-0.033)
LST gradients.

**Interpretation.** The bright-is-cooler cross-sectional pattern is not an
artefact of surface skin temperature -- it shows up in air temperature too. Same
confounding story: `glass_ta_mean` goes into the regression notebook as a
robustness outcome (Section R5 there).

In [ ]:
fig = viz.plot_distribution(con, "glass_ta_mean", table="panel_glass", groupby="mine_ever",
                            kind="ecdf", bins=200)
fig.axes[0].set_title("GLASS air temp by mine exposure (ever within 20 km)")
fig

**Result.** Mine-ever pixels are cooler in air temperature too, mirroring the
~3.6 K daytime-LST gap from Section 4.

**Interpretation.** The treated / control imbalance is a property of *where*
mines are, not of which temperature product you use -- it survives the swap to
air temperature, so pixel FE remain essential.

### Raw VIIRS median (`viirs_annual_median`)

In [ ]:
fig = viz.plot_distribution(con, "viirs", table="panel_viirs", transform="log1p", offset=1.0,
                            bins=60, logy=True)
fig.axes[0].set_title("log(viirs_annual_median + 1)  --  2013-2021 (log count axis)")
fig

**Result.** VIIRS is available 2013-2021 only (2012 sits on a different scale and
is dropped). 57 % of pixel-years are <= 0 (background subtraction can go
negative), 85 % <= 1.

**Interpretation.** Sparser and noisier than `ntl_harm`, and it covers less than
a third of the study period -- a robustness check on the VIIRS window, not a
drop-in replacement.

In [ ]:
fig = viz.plot_binscatter(con, y=Y, x="viirs", x_transform="log1p", offset=1.0,
                          table="panel_viirs", bins=30)
fig.axes[0].set_title("lst_day_mean vs log(viirs + 1), 2013-2021  -- alternative NTL measure")
fig

**Result.** Same downward gradient as with `ntl_harm`. On the overlapping window
`corr(log1p(ntl_harm), log1p(viirs)) = 0.74` and `corr(ntl_harm, viirs) = 0.49`.

**Interpretation.** The harmonized series and the raw sensor broadly agree, and
the OLS relationship holds with either -- so the section-3 result is not an
artefact of the DMSP/VIIRS harmonization.

In [ ]:
# How well does the harmonized ntl_harm track raw VIIRS on the overlap?
fig = viz.plot_binscatter(con, y="viirs", x=X, y_transform="log1p", x_transform="log1p",
                          offset=1.0, table="panel_viirs", bins=30)
fig.axes[0].set_title("log(viirs+1) vs log1p(ntl_harm), 2013-2021  -- do the two lights series agree?")
fig

**Result.** Tight positive relationship: `corr(log1p(ntl_harm), log(viirs+1))`
~ 0.79 on 2013-2020 (2012 excluded -- it sits on a different DN scale, mean 73 vs
~0.35). The binned means rise monotonically.

**Interpretation.** `ntl_harm` is a faithful stand-in for raw VIIRS where they
overlap, so using the harmonized series for the full 1992-2022 window (rather
than restricting to the VIIRS years) is defensible -- the sensor swap is the
weak point, not the harmonization per se.

## 9. Data-quality filters -- valid observation count and gas flares

Two filters that change the analysis sample: dropping pixel-years with too few
valid daytime-LST observations, and dropping NTL contaminated by gas flares.

In [ ]:
fig = viz.plot_distribution(con, "valid_day", table="panel", bins=60, logy=True, vlines=VALID_MIN)
fig.axes[0].set_title(f"valid_month_count_day_annual  (log count axis, dashed line at {VALID_MIN})")
fig

**Result.** The annual valid-observation count for daytime LST runs 0-1200
(looks like ~months x 100; p50 ~ 1104). **~11 % of non-null pixel-years fall
below 300, ~4 % below 100.** Where the count is very low, `lst_day_mean` itself
is sometimes missing (92.5 % present in the 0-100 bucket vs 100 % elsewhere).

**Interpretation.** A handful of pixel-years rest on very thin coverage. A cut at
`VALID_MIN = 300` is a defensible screen; the exact unit is not documented, so
treat it as a percentile-style threshold and check sensitivity below.

In [ ]:
# Sample share and LST moments vs the valid-count cutoff.
vcuts = [0, 100, 200, 300, 500, 700, 900]
vsens = con.execute(f"""
    SELECT c,
           avg((valid_day >= c)::INT)                              AS share,
           avg(lst_day_mean)     FILTER (WHERE valid_day >= c)     AS lst_mean,
           stddev_samp(lst_day_mean) FILTER (WHERE valid_day >= c) AS lst_sd
    FROM panel CROSS JOIN (SELECT unnest({vcuts}) AS c) g
    WHERE valid_day IS NOT NULL
    GROUP BY c ORDER BY c
""").fetchdf()

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(vsens.c, vsens.share, "o-"); ax[0].axvline(VALID_MIN, ls="--", c="0.4")
ax[0].set(xlabel="valid-count cutoff", ylabel="share of pixel-years kept")
ax[1].plot(vsens.c, vsens.lst_mean, "o-", label="mean"); ax[1].axvline(VALID_MIN, ls="--", c="0.4")
ax[1].plot(vsens.c, vsens.lst_sd, "s--", label="sd")
ax[1].legend(); ax[1].set(xlabel="valid-count cutoff", ylabel="lst_day_mean among kept rows")
fig.tight_layout(); fig

**Result.** Raising the cutoff drops the retained mean LST (low-count pixel-years
are hot: ~305.7 K in the 0-100 bucket vs 302.5 K for 600+) and **raises** its sd
(7.6 K -> 14.5 K) -- the dropped rows are hot, spatially uniform regions
(persistent clear-sky deserts / tropics).

**Interpretation.** The valid-count filter is not neutral: it trims a
non-random, low-variance slice, so it modestly shifts the outcome distribution.
Report the core spec with and without it; `VALID_MIN = 300` removes ~11 % of
rows.

In [ ]:
flare_tab = con.execute("""
    SELECT (flare_band > 0) AS flare, count(*) AS n,
           avg(ntl_harm) AS ntl_mean, quantile_cont(ntl_harm, 0.5) AS ntl_median,
           avg(lst_day_mean) AS lst_mean
    FROM panel WHERE flare_band IS NOT NULL
    GROUP BY flare ORDER BY flare
""").fetchdf()
display(flare_tab)

fig = viz.plot_distribution(con, X, table="panel_flareable", groupby="is_flare",
                            kind="ecdf", bins=300)
fig.axes[0].set_title("ntl_harm by gas-flare flag (VIIRS-era pixel-years)")
fig

**Result.** The flare mask covers VIIRS-era rows only (12.1 M of 37.4 M).
**1.1 % are flagged** (`flare_band > 0`, 134.6 k rows). Flagged pixel-years have
~6x the NTL of the rest (mean 18.8 vs 3.2, median 10.9 vs 0.0) and sit squarely
in the bright tail; LST is marginally cooler (302.0 vs 303.5 K).

**Interpretation.** Few rows, but high leverage -- they are exactly the
high-`log(ntl)` observations a log-NTL regression loads on, and gas flares are
not human economic activity. Keep only `flare_band = 0`. Caveat: no mask before
2012, so pre-VIIRS flare contamination is unaddressed.

In [ ]:
fig = viz.plot_binscatter(con, y=Y, x=X, x_transform="log1p", groupby="is_flare",
                          table="panel_flareable", bins=20)
fig.axes[0].set_title("lst_day_mean vs log1p(ntl_harm), by flare flag (VIIRS-era)")
fig

**Result.** The flare-flagged bins cluster at high `log1p(ntl_harm)` and lie
slightly below the non-flare line.

**Interpretation.** Including flares adds a small group of very bright, slightly
cooler leverage points at the top of the NTL range, steepening the apparent
slope. Dropping them is the conservative choice and mainly affects the intensive
margin.

## What the descriptives imply for the core analysis

- **Outcome** (`lst_day_mean`) is clean; 32.5 % missing (MODIS gaps) bounds the
  sample.
- **Regressor** (`ntl_harm`) is 68 % exact zeros. The **extensive/intensive
  split** is well-behaved -- `log(ntl_harm) | ntl_harm >= 7` is near-symmetric,
  ~20 % of pixels switch the threshold -- but the threshold indicator is
  contaminated by the 2013 DMSP -> VIIRS transition; drop 2013-2014 or add a
  sensor-era term, and check robustness to `NTL_HI`.
- **Count instrument** (`mine_count_20km`) is zero for 95 % of pixel-years; first
  stage positive but weak (raw r ~ 0.07, within r ~ 0.04).
- **Price-shock instrument** (`mine_priceshock_20km`) is even sparser (80 % zero
  within mine pixels) and has ~zero raw correlation with NTL -- an
  over-identifying check, not a primary instrument.
- **Confounding is severe in levels**: mine pixels are ~3.6 K cooler and the raw
  LST-NTL gradient is negative. Pixel + year FE are essential; the exclusion
  restriction (mines -> LST only via NTL) is the key vulnerability.
- **Robustness holds in sign**: the negative NTL-LST gradient survives with
  night-time LST (weaker -- pointing at a land-cover, not heat-retention,
  channel), with **GLASS 2 m air temperature** (2000-2020; `corr` with night LST
  0.98), and with raw VIIRS on 2013-2021 (`ntl_harm` and VIIRS agree at
  r ~ 0.79 in logs).
- **Data-quality filters are not neutral**: dropping low-valid-count pixel-years
  trims a hot, low-variance slice and shifts the outcome distribution; the
  gas-flare mask (VIIRS-era, 1.1 % of rows) removes few but high-leverage bright
  observations. Run the core spec with and without each (`VALID_MIN`,
  `flare_band = 0`).
- **Next steps**: full-panel `duckreg` 2SLS with first-stage F and weak-IV-robust
  CIs, for both the level spec and the extensive/intensive split; price-shock as
  an over-ID test; biome / country-by-year controls to probe exclusion. Fitted
  models and their diagnostics live in a separate notebook.

In [ ]:
con.close()